In [1]:
import pandas as pd
import numpy as np

import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
lol = pd.read_csv("2022_LoL_esports_match_data_from_OraclesElixir.csv")
lol.head()

/var/folders/cz/04thlfvs2333xz77pw9z44jh0000gn/T/ipykernel_83852/2061346798.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  lol = pd.read_csv("2022_LoL_esports_match_data_from_OraclesElixir.csv")


,gameid,datacompleteness,url,league,year,split,playoffs,date,game,patch,...,opp_csat25,golddiffat25,xpdiffat25,csdiffat25,killsat25,assistsat25,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
0,ESPORTSTMNT01_2690210,complete,NaN,LCKC,2022,Spring,0,2022-01-10 07:44:08,1,12.01,...,203.0,605.0,-525.0,9.0,0.0,1.0,1.0,0.0,2.0,0.0
1,ESPORTSTMNT01_2690210,complete,NaN,LCKC,2022,Spring,0,2022-01-10 07:44:08,1,12.01,...,163.0,421.0,-903.0,-28.0,2.0,4.0,2.0,1.0,5.0,1.0
2,ESPORTSTMNT01_2690210,complete,NaN,LCKC,2022,Spring,0,2022-01-10 07:44:08,1,12.01,...,187.0,-149.0,-224.0,-5.0,1.0,3.0,0.0,3.0,4.0,3.0
3,ESPORTSTMNT01_2690210,complete,NaN,LCKC,2022,Spring,0,2022-01-10 07:44:08,1,12.01,...,284.0,-1288.0,-2005.0,-85.0,2.0,1.0,2.0,3.0,4.0,0.0
4,ESPORTSTMNT01_2690210,complete,NaN,LCKC,2022,Spring,0,2022-01-10 07:44:08,1,12.01,...,27.0,499.0,-314.0,12.0,1.0,3.0,2.0,0.0,7.0,2.0


In [3]:
lol.shape
lol.columns.tolist()

['gameid',
 'datacompleteness',
 'url',
 'league',
 'year',
 'split',
 'playoffs',
 'date',
 'game',
 'patch',
 'participantid',
 'side',
 'position',
 'playername',
 'playerid',
 'teamname',
 'teamid',
 'firstPick',
 'champion',
 'ban1',
 'ban2',
 'ban3',
 'ban4',
 'ban5',
 'pick1',
 'pick2',
 'pick3',
 'pick4',
 'pick5',
 'gamelength',
 'result',
 'kills',
 'deaths',
 'assists',
 'teamkills',
 'teamdeaths',
 'doublekills',
 'triplekills',
 'quadrakills',
 'pentakills',
 'firstblood',
 'firstbloodkill',
 'firstbloodassist',
 'firstbloodvictim',
 'team kpm',
 'ckpm',
 'firstdragon',
 'dragons',
 'opp_dragons',
 'elementaldrakes',
 'opp_elementaldrakes',
 'infernals',
 'mountains',
 'clouds',
 'oceans',
 'chemtechs',
 'hextechs',
 'dragons (type unknown)',
 'elders',
 'opp_elders',
 'firstherald',
 'heralds',
 'opp_heralds',
 'void_grubs',
 'opp_void_grubs',
 'firstbaron',
 'barons',
 'opp_barons',
 'atakhans',
 'opp_atakhans',
 'firsttower',
 'towers',
 'opp_towers',
 'firstmidtower',


In [4]:
team = lol[lol["position"] == "team"].copy()
team.shape

(25058, 165)

In [5]:
team[["gameid", "side", "teamname", "result"]].head()

,gameid,side,teamname,result
10,ESPORTSTMNT01_2690210,Blue,HANJIN BRION Challengers,0
11,ESPORTSTMNT01_2690210,Red,Nongshim Esports Academy,1
22,ESPORTSTMNT01_2690219,Blue,T1 Esports Academy,0
23,ESPORTSTMNT01_2690219,Red,Liiv SANDBOX Youth,1
34,8401-8401_game_1,Blue,Oh My God,1


In [13]:
# Hypothesis Test

blue = team[team["side"] == "Blue"]

observed_blue_win_rate = blue["result"].mean()
observed_stat = observed_blue_win_rate - 0.5

observed_blue_win_rate, observed_stat

(np.float64(0.5247825045893527), np.float64(0.024782504589352716))

In [14]:
n_blue_games = blue.shape[0]

n_repetitions = 10000
simulated_stats = np.array([])

for i in range(n_repetitions):
    simulated_results = np.random.choice([0, 1], size=n_blue_games, p=[0.5, 0.5])
    simulated_blue_win_rate = simulated_results.mean()
    simulated_stat = simulated_blue_win_rate - 0.5
    simulated_stats = np.append(simulated_stats, simulated_stat)

p_value = np.mean(simulated_stats >= observed_stat)

observed_blue_win_rate, observed_stat, p_value

(np.float64(0.5247825045893527),
 np.float64(0.024782504589352716),
 np.float64(0.0))

In [9]:
#Q2
# Baseline Model

model_cols = [
    "kills",
    "deaths",
    "assists",
    "earnedgold",
    "totalgold",
    "dragons",
    "barons",
    "towers"
]

available_model_cols = [col for col in model_cols if col in team.columns]
available_model_cols

['kills',
 'deaths',
 'assists',
 'earnedgold',
 'totalgold',
 'dragons',
 'barons',
 'towers']

In [10]:
X = team[available_model_cols]
y = team["result"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, available_model_cols)
])

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, y_pred)
baseline_accuracy

0.9816440542697525

In [11]:
print("Baseline accuracy:", baseline_accuracy)
print()
print(classification_report(y_test, y_pred))

Baseline accuracy: 0.9816440542697525

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      3133
           1       0.98      0.99      0.98      3132

    accuracy                           0.98      6265
   macro avg       0.98      0.98      0.98      6265
weighted avg       0.98      0.98      0.98      6265



In [12]:
confusion_matrix(y_test, y_pred)

array([[3061,   72],
       [  43, 3089]])